In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

def Lag_3(X):
    L_0 = np.exp(-X / 2)
    L_1 = np.exp(-X / 2) * (1 - X)
    L_2 = np.exp(-X / 2) * (1 - 2 * X + X ** 2 / 2)
    return np.column_stack((L_0, L_1, L_2))



def LSM(S0, K, r, sd, T, I, M):
    dt = 1/M
    M=int(M*T)
    df = np.exp(-r*dt)
    S = np.zeros((I, M + 1))
    S[:, 0] = S0  
    S[:, 1:] = S0 * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(I, M)), axis=1)) 


    s = S/K 
    h = np.maximum(1 - s, 0)
    h[:, 0] = 0 
    h_copy = h.copy()

    for i in range(M, 1, -1):
        ITM = h[:, i - 1] > 0

        Y = df * h_copy[ITM, i]
        X = s[ITM, i - 1]
        Z = Lag_3(X)

        reg = LinearRegression().fit(Z, Y)

        basis = Lag_3(s[:, i - 1])
        cont = reg.predict(basis)

        h[:, i - 1] = np.where(h[:, i - 1] <= cont, 0, h[:, i - 1])
        h[h[:, i - 1] > 0, i:] = 0
        h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * df, h_copy[:, i - 1])

    df_vec = np.power(df, np.arange(M + 1)).reshape(-1, 1)

    disc_CashFlows = h @ df_vec

    price = sum(disc_CashFlows) * K / I
    se = np.sqrt(sum((disc_CashFlows * K - price) ** 2)) / I


    return price, se

LSM(36,40,0.06,0.2,1,10**5,50)

(array([4.4874941]), array([0.00925541]))